## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
    "requests",  # OllamaClient에서 사용
)

# Ollama 서버 연결 확인
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama connected ✓ models: {models}")
except Exception:
    print("⚠️ Ollama not available — LLM 기능은 규칙 기반으로 폴백됩니다")


# 모델 비교 실험 — 크기별 성능 트레이드오프

이 노트북은 같은 workflow에 서로 다른 모델 크기(Qwen3.5-9B, 4B, 2B)와 규칙 기반 경로를 넣었을 때, 답변 품질과 지연 시간(latency), 그리고 GPU 메모리 사용량이 어떻게 달라지는지 비교한다. 모델 크기 선택은 단순히 "큰 모델이 더 좋다"의 문제가 아니라, 어떤 목적에 어떤 비용을 감수할지의 문제다.

## 학습 목표
- 모델 크기별 품질-속도-자원 사용 트레이드오프를 읽을 수 있다.
- 단일 질문 비교와 소규모 배치 평가가 왜 서로 다른 인사이트를 주는지 이해한다.
- `evaluate_system` 계열 평가 로직을 모델 비교 실험에 어떻게 재사용할 수 있는지 설명할 수 있다.
- DGX 환경에서 `nvidia-smi` 스냅샷을 해석하는 기본 감각을 익힌다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RuntimeConfig
from src.llm_client import OllamaClient

runtime_config = RuntimeConfig.auto_detect()
inspector_client = OllamaClient()
llm_live = inspector_client.is_available()
available_models = inspector_client.list_models() if llm_live else []

print(sys.executable)
{
    'device': runtime_config.device,
    'llm_available': runtime_config.llm_available,
    'configured_model': runtime_config.llm_model,
    'available_models': available_models,
}

## 실험 설계

**목적**
- 비교 대상 모델과 샘플 질문 집합을 먼저 고정해 실험 결과를 해석 가능하게 만든다.

**핵심 로직**
- `demo` 데이터 프로필에서 query type별 2개씩 총 10개 질문을 뽑는다.
- 이 10개는 simple_lookup, comparison, multi_hop, summary, insufficient_evidence_risk를 모두 포함해 한쪽 유형으로 치우치지 않게 한다.
- 이후 단일 질문 비교와 전체 평가 모두 이 샘플 질문 묶음을 기준으로 읽는다.

**주요 파라미터**
- `variant`: `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b`, `rule_based`
- `sampled_questions`: query type별 2개씩 뽑은 비교용 mini benchmark
- `path_mode`: 실제 LLM live 호출인지, fallback인지, server unavailable인지 표시하는 상태값

이 설계가 중요한 이유는, 모델 비교는 질문 난이도에 매우 민감하기 때문이다. 쉬운 질문 몇 개만 고르면 작은 모델도 충분히 좋아 보이고, 너무 어려운 질문만 고르면 모든 모델이 나빠 보일 수 있다.

### 데이터 복잡도까지 포함한 실험 설계

이번에는 `demo`와 `tech_docs`를 둘 다 로드한다. `demo`는 빠르게 sanity check를 하기 좋은 작은 코퍼스이고, `tech_docs`는 모델 차이가 실제로 드러나는 큰 코퍼스다. 그래서 이 노트북은 "모델 크기 비교"만이 아니라 "모델 크기 × 데이터 복잡도" 비교를 함께 보는 구조가 된다.

즉, 여기서 고정하는 것은 query type 분포이고, 바꾸는 것은 두 가지다.
- 모델 크기: 9B / 4B / 2B / rule_based
- 데이터 복잡도: demo / tech_docs


In [ ]:
import pandas as pd
from IPython.display import display

from src.data_profiles import load_profile


def sample_questions(profile: dict) -> pd.DataFrame:
    frame = pd.DataFrame(profile['eval_dataset']).sort_values(['question_type', 'id'])
    return (
        frame.groupby('question_type', as_index=False, group_keys=False)
        .head(2)
        .reset_index(drop=True)
    )


demo_profile = load_profile('demo', persist=False)
tech_profile = load_profile('tech_docs', persist=False)

demo_sampled_questions = sample_questions(demo_profile)
tech_sampled_questions = sample_questions(tech_profile)

comparison_profiles = {
    'demo': {
        'profile': demo_profile,
        'retriever': demo_profile['retriever'],
        'samples': demo_sampled_questions,
    },
    'tech_docs': {
        'profile': tech_profile,
        'retriever': tech_profile['retriever'],
        'samples': tech_sampled_questions,
    },
}

profile = tech_profile
retriever = profile['retriever']
sampled_questions = tech_sampled_questions

stats_frame = pd.DataFrame(
    [
        {
            'dataset': name,
            'document_count': bundle['profile']['stats']['document_count'],
            'chunk_count': bundle['profile']['stats']['chunk_count'],
            'eval_question_count': bundle['profile']['stats']['eval_question_count'],
        }
        for name, bundle in comparison_profiles.items()
    ]
)
display(stats_frame)
print(f"Primary profile: tech_docs ({profile['stats']})")
print(f"Sampled questions: demo={len(demo_sampled_questions)}, tech_docs={len(tech_sampled_questions)}")
sampled_questions[['id', 'question_type', 'question', 'expected_status']]


## 실험: 단일 질문 비교

**목적**
- 같은 질문 하나에 대해 네 가지 경로가 어떤 스타일과 상태를 보이는지 직관적으로 확인한다.

**핵심 로직**
- `LLMConfig(model=...)`로 모델명을 덮어써 같은 코드 경로에서 다른 Ollama 모델을 호출한다.
- `run_workflow(use_llm=True/False)`를 동일 질문에 반복 적용해 최종 상태, grounding 여부, latency를 나란히 놓는다.

**주요 파라미터**
- `single_question`: 비교 대상 질문이다.
- `single_variants`: `(모델명, use_llm 여부, client)` 튜플 목록이다.
- `path_mode`: 실제 live 호출인지 fallback인지 구분하는 해석용 필드다.

**결과 해석 가이드**
- `answer_preview`는 말투와 압축 정도를 보는 용도다. 품질 판단은 `grounded`, `coverage_score`, `final_status`를 같이 봐야 한다.
- `warning_count`가 있으면 해당 행은 모델 자체 성능이 아니라 fallback 경로 영향을 받았을 가능성이 크다.
- `latency_seconds`는 한 질문 기준 체감 지연을 보여 준다. 사용자 경험 관점에서는 이 숫자가 매우 중요하다.

**💡 면접 포인트**
- 모델 크기 비교는 평균표만 보면 놓치는 차이가 많아서, 단일 질문 질적 비교를 먼저 보는 편이 좋다.
- 같은 workflow contract 위에서 모델만 바꾸면, 모델 효과와 시스템 효과를 분리해 설명하기 쉬워진다.

### src 코드 펼침: `OllamaClient.chat()`의 모델 오버라이드

```python
def chat(self, messages: list[dict[str, str]], **kwargs: Any) -> str:
    payload = {
        "model": kwargs.get("model", self.config.model),
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": kwargs.get("temperature", self.config.temperature),
            "num_predict": kwargs.get("max_tokens", self.config.max_tokens),
        },
    }
    response = self.session.post(...)
    response.raise_for_status()
    return response.json()["message"]["content"]
```

- `kwargs.get("model", self.config.model)` 한 줄이 모델 비교 실험의 핵심이다. client를 새로 만들지 않아도 호출마다 모델만 바꿔 끼울 수 있다.
- 예를 들어 기본 config는 `qwen3.5:9b`로 두고, 어떤 셀에서는 `client.chat(messages, model="qwen3.5:4b")`처럼 덮어쓸 수 있다.
- 같은 질문, 같은 workflow, 같은 retriever를 유지한 채 모델 이름만 바꿔 호출하면 비교 실험이 공정해진다.
- 이 프로젝트가 모델 비교를 "별도 코드베이스"가 아니라 같은 함수 호출 인터페이스 안에서 수행할 수 있는 이유가 여기 있다.

### 왜 단일 질문은 `tech_docs`로 보나

단일 질문 비교는 모델 크기 차이가 실제 문장 품질로 어떻게 드러나는지 보는 셀이다. 이 차이는 `demo`보다 `tech_docs`에서 더 선명하게 나타난다. 작은 코퍼스에서는 2B도 정답 문장을 거의 그대로 뽑아낼 수 있지만, 실제 기술 문서에서는 개념 연결과 요약 압축 능력이 더 중요해지기 때문이다.


In [ ]:
import time
import warnings

import pandas as pd

from src.llm_client import LLMConfig, OllamaClient
from src.workflow import run_workflow

single_sample = tech_sampled_questions.iloc[0].to_dict()
single_question = single_sample['question']

single_variants = [
    ('qwen3.5:9b', True, OllamaClient(LLMConfig(model='qwen3.5:9b'))),
    ('qwen3.5:4b', True, OllamaClient(LLMConfig(model='qwen3.5:4b'))),
    ('qwen3.5:2b', True, OllamaClient(LLMConfig(model='qwen3.5:2b'))),
    ('rule_based', False, None),
]

rows = []
for variant_name, use_llm, client in single_variants:
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        state = run_workflow(
            single_question,
            retriever=retriever,
            use_llm=use_llm,
            llm_client=client,
        )
    latency = round(time.perf_counter() - start, 4)
    path_mode = 'rule_based'
    if use_llm:
        if not llm_live:
            path_mode = 'server_unavailable'
        elif caught:
            path_mode = 'fallback'
        else:
            path_mode = 'live'
    rows.append(
        {
            'dataset': 'tech_docs',
            'variant': variant_name,
            'path_mode': path_mode,
            'final_status': state['final_status'],
            'grounded': state['verification_result'].is_grounded,
            'coverage_score': state['verification_result'].coverage_score,
            'latency_seconds': latency,
            'answer_preview': ' '.join(state['final_answer'].split())[:180],
            'warning_count': len(caught),
        }
    )

print(single_question)
pd.DataFrame(rows)


## 구현: 전체 평가 실행

**목적**
        - 단일 질문 비교를 넘어, 10개 질문 전체에 대해 모델별 평균 품질 지표를 계산한다.

        **핵심 로직**
        - 이 notebook은 `src/evaluator.py`의 지표 계산 함수들을 부분적으로 재사용한다.
        - 실제 `evaluate_system()`은 dataset 전체를 baseline/agent 두 시스템으로 비교하는 함수다. 여기서는 그 로직을 변형해 model variant별로 반복 실행한다.

        **실제 소스 코드: score_answer() + retrieval_hit_rate()**
        ```python
        def score_answer(predicted_answer: str, gold_answer: str, predicted_status: str, expected_status: str) -> float:
    if expected_status == "abstained":
        return 1.0 if predicted_status == "abstained" else 0.0
    if predicted_status == "abstained":
        return 0.0
    return token_f1(predicted_answer, gold_answer)

        def retrieval_hit_rate(retrieved_docs: list[dict[str, Any]], expected_sources: list[str]) -> float:
    if not expected_sources:
        return 1.0 if retrieved_docs else 0.0
    sources = {doc["source"] for doc in retrieved_docs}
    return 1.0 if any(source in sources for source in expected_sources) else 0.0
        ```

        **실제 소스 코드: evaluate_system()**
        ```python
        def evaluate_system(
    system: SystemName,
    dataset: list[dict[str, Any]] | None = None,
    trace_dir: Path | None = None,
    repeats: int = 3,
) -> pd.DataFrame:
    evaluation_set = dataset or load_eval_dataset()
    retriever = build_demo_index(persist=False)
    rows: list[dict[str, Any]] = []

    for run_id in range(1, repeats + 1):
        for sample in evaluation_set:
            start = time.perf_counter()
            if system == "baseline":
                result = run_baseline_rag(sample["question"], retriever)
                predicted_question_type = None
                grounding_pass = False
            else:
                trace_path = None
                if trace_dir is not None:
                    trace_path = trace_dir / f"{sample['id']}_run{run_id}.json"
                result = run_workflow(sample["question"], retriever, trace_path=trace_path)
                predicted_question_type = result["query_type"]
                grounding_pass = result["verification_result"].is_grounded

            latency = time.perf_counter() - start
            record = EvaluationRecord(
                system=system,
                question_id=sample["id"],
                run_id=run_id,
                question=sample["question"],
                expected_question_type=sample["question_type"],
                predicted_question_type=predicted_question_type,
                expected_status=sample.get("expected_status", "answered"),
                predicted_status=result["final_status"],
                final_answer=result["final_answer"],
                answer_correctness=score_answer(
                    result["final_answer"],
                    sample["gold_answer"],
                    result["final_status"],
                    sample.get("expected_status", "answered"),
                ),
                retrieval_hit_rate=retrieval_hit_rate(result["retrieved_docs"], sample.get("expected_sources", [])),
                grounding_pass=grounding_pass,
                abstained=result["final_status"] == "abstained",
                abstain_precision=0.0,
                latency_seconds=round(latency, 4),
                average_steps=float(len(result.get("trace", []))),
                failure_type="",
            ).to_dict()
            record["expected_sources"] = sample.get("expected_sources", [])
            record["trace"] = result.get("trace", [])
            record["errors"] = result.get("errors", [])
            record["citations"] = [
                citation.get("source", "")
                for citation in result.get("citations", [])
                if isinstance(citation, dict) and citation.get("source")
            ]
            record["failure_type"] = classify_failure(record)
            record["abstain_precision"] = 1.0 if (
                record["abstained"] and record["expected_status"] == "abstained"
            ) else 0.0
            rows.append(record)

    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame["grounding_pass_rate"] = frame["grounding_pass"].astype(float)
    else:
        frame["grounding_pass_rate"] = []
    return frame
        ```

        **코드 읽기 포인트**
        - `score_answer()`는 abstain 정답 여부를 먼저 처리하고, answered 질문은 token F1으로 본다.
        - `retrieval_hit_rate()`는 expected source가 top-k에 들어왔는지만 보므로, answer quality와 분리된 retrieval 진단 지표다.
        - `evaluate_system()`은 record 단위로 metrics를 저장하는 패턴이라, 이 notebook처럼 다른 variant 비교로 확장하기 쉽다.
        - 이번 notebook의 `model_results`는 사실상 `evaluate_system()`의 notebook 버전이라고 볼 수 있다.

        **주요 파라미터**
        - `model_variants`: 비교할 모델 목록
        - `parameter_scale`: 9B/4B/2B/0B(규칙 기반)처럼 모델 크기를 읽기 쉽게 붙인 label
        - `live_rate`: 실제 live LLM 경로가 얼마나 자주 사용됐는지 보여 주는 안정성 지표

        **결과 해석 가이드**
        - `answer_correctness`는 답변 품질, `retrieval_hit_rate`는 retrieval 안정성, `grounding_pass_rate`는 verifier 기준 신뢰도를 뜻한다.
        - `average_steps`는 workflow 구조가 같다면 거의 비슷해야 한다. 크게 다르면 fallback이나 예외 흐름이 섞였을 가능성이 있다.
        - `live_rate`가 낮으면 해당 모델 결과는 모델 품질보다 인프라 가용성에 더 큰 영향을 받았다고 해석해야 한다.

### src 코드 펼침: `chat_with_metadata()`

```python
def chat_with_metadata(self, messages: list[dict[str, str]], **kwargs: Any) -> dict[str, Any]:
    payload = {...}
    response = self.session.post(...)
    response.raise_for_status()
    data = response.json()
    return {
        "content": data["message"]["content"],
        "model": data.get("model", self.config.model),
        "eval_count": data.get("eval_count", 0),
        "eval_duration_ms": data.get("eval_duration", 0) / 1_000_000,
        "total_duration_ms": data.get("total_duration", 0) / 1_000_000,
    }
```

- `eval_count`는 생성 토큰 수라서 "큰 모델이 항상 더 길게 말하는가"를 보는 데 도움을 준다.
- `eval_duration_ms`는 모델이 실제로 디코딩에 쓴 시간이다. 네트워크나 파이썬 오버헤드보다 모델 자체 속도 차이를 읽기에 좋다.
- `total_duration_ms`는 사용자가 체감하는 전체 요청 시간이다. 운영 환경에서 SLA를 볼 때는 이 값이 더 중요하다.

### src 코드 펼침: `score_answer()`

```python
def score_answer(predicted_answer: str, gold_answer: str, predicted_status: str, expected_status: str) -> float:
    if expected_status == "abstained":
        return 1.0 if predicted_status == "abstained" else 0.0
    if predicted_status == "abstained":
        return 0.0
    return token_f1(predicted_answer, gold_answer)
```

- 이 함수는 정답 문자열이 완전히 똑같은지(exact match)를 보지 않는다. 대신 `token_f1()`로 토큰 겹침의 precision/recall 균형을 본다.
- 예를 들어 gold answer가 "pilot window spans 25 days"이고 예측이 "the pilot lasts 25 days"라면 exact match는 실패지만 token F1은 부분 점수를 줄 수 있다.
- `expected_status == "abstained"`일 때는 오히려 답하지 않는 것이 정답이다. 즉, 이 프로젝트의 평가는 "무조건 많이 답하는 시스템"을 칭찬하지 않는다.

### src 코드 펼침: `run_workflow()`의 `use_llm` / `llm_client` 전달 구조

```python
def run_workflow(
    query: str,
    retriever: Any,
    top_k: int = DEFAULT_TOP_K,
    trace_path: Path | None = None,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> AgentState:
    state = create_initial_state(query)
    effective_llm_client = llm_client or (OllamaClient() if use_llm else None)

    _execute_workflow_steps(
        state,
        WORKFLOW_STEPS,
        retriever=retriever,
        top_k=top_k,
        use_llm=use_llm,
        llm_client=effective_llm_client,
    )
```

- 이 구조 덕분에 notebook에서는 `LLMConfig(model="qwen3.5:4b") -> OllamaClient(config) -> run_workflow(..., use_llm=True, llm_client=client)` 흐름을 그대로 반복하면 된다.
- 즉, workflow 로직은 고정하고 모델만 바꾸는 실험이 가능하다. 실험 설계에서 가장 중요한 것은 나머지 변수를 최대한 고정하는 것이다.
- 규칙 기반 경로는 `use_llm=False` 하나로 비교군이 된다. 같은 함수 시그니처로 LLM 경로와 non-LLM 경로를 모두 태울 수 있다는 점이 실험 자동화에 유리하다.

### 모델 × 데이터 조합 평가

이제부터는 `demo + 9B`, `demo + 4B`, `demo + 2B`, `demo + rule_based`, `tech_docs + 9B`, `tech_docs + 4B`, `tech_docs + 2B`, `tech_docs + rule_based`를 한꺼번에 비교한다. 이렇게 해야 "9B가 항상 최고인가?"가 아니라 "어떤 데이터에서 9B의 추가 비용이 정당화되는가?"를 말할 수 있다.


In [ ]:
import time
import warnings

import pandas as pd
from IPython.display import display

from src.evaluator import retrieval_hit_rate, score_answer
from src.llm_client import LLMConfig, OllamaClient
from src.workflow import run_workflow

model_variants = [
    ('qwen3.5:9b', True),
    ('qwen3.5:4b', True),
    ('qwen3.5:2b', True),
    (None, False),
]

parameter_scale = {
    'qwen3.5:9b': '9B',
    'qwen3.5:4b': '4B',
    'qwen3.5:2b': '2B',
    'rule_based': '0B',
}

raw_rows = []
for dataset_name, bundle in comparison_profiles.items():
    retriever = bundle['retriever']
    samples = bundle['samples'].to_dict(orient='records')

    for model_name, use_llm in model_variants:
        variant_label = model_name or 'rule_based'
        client = OllamaClient(LLMConfig(model=model_name)) if use_llm and model_name else None

        for sample in samples:
            start = time.perf_counter()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                state = run_workflow(
                    sample['question'],
                    retriever=retriever,
                    use_llm=use_llm,
                    llm_client=client,
                )
            latency = round(time.perf_counter() - start, 4)

            if not use_llm:
                path_mode = 'rule_based'
            elif not llm_live:
                path_mode = 'server_unavailable'
            elif caught:
                path_mode = 'fallback'
            else:
                path_mode = 'live'

            raw_rows.append(
                {
                    'dataset': dataset_name,
                    'variant': variant_label,
                    'parameter_scale': parameter_scale[variant_label],
                    'question_id': sample['id'],
                    'question_type': sample['question_type'],
                    'question': sample['question'],
                    'path_mode': path_mode,
                    'predicted_status': state['final_status'],
                    'answer_correctness': score_answer(
                        state['final_answer'],
                        sample['gold_answer'],
                        state['final_status'],
                        sample.get('expected_status', 'answered'),
                    ),
                    'retrieval_hit_rate': retrieval_hit_rate(
                        state['retrieved_docs'],
                        sample.get('expected_sources', []),
                    ),
                    'grounding_pass_rate': float(state['verification_result'].is_grounded),
                    'latency_seconds': latency,
                    'average_steps': float(len(state['trace'])),
                    'abstained': state['final_status'] == 'abstained',
                    'should_abstain': sample.get('expected_status', 'answered') == 'abstained',
                }
            )

model_results = pd.DataFrame(raw_rows)
model_summary = (
    model_results.groupby(['dataset', 'variant', 'parameter_scale'], as_index=False)
    .agg(
        answer_correctness=('answer_correctness', 'mean'),
        retrieval_hit_rate=('retrieval_hit_rate', 'mean'),
        grounding_pass_rate=('grounding_pass_rate', 'mean'),
        latency_seconds=('latency_seconds', 'mean'),
        average_steps=('average_steps', 'mean'),
        live_rate=('path_mode', lambda s: float((s == 'live').mean())),
    )
    .round(3)
)

comparison_gap = (
    model_summary.pivot(index='variant', columns='dataset', values='answer_correctness')
    .reset_index()
)
if {'demo', 'tech_docs'}.issubset(comparison_gap.columns):
    comparison_gap['tech_minus_demo'] = (comparison_gap['tech_docs'] - comparison_gap['demo']).round(3)

display(model_summary)
comparison_gap


## 결과 해석: 시각화로 읽는 모델 선택 기준

막대 그래프와 radar chart는 서로 다른 역할을 한다. 막대 그래프는 절대값 비교에 좋고, radar chart는 여러 metric의 균형을 한 눈에 보여준다.

읽는 순서는 아래를 권한다.
- `answer_correctness`와 `grounding_pass_rate`로 품질과 안전성의 기본 수준을 본다.
- `latency_seconds`와 `speed_score`로 응답 비용을 본다.
- 마지막에 `live_rate`를 함께 봐서, 좋은 숫자가 실제 모델 성능인지 단순 fallback 회피 덕분인지 판단한다.

특히 radar chart는 면적이 크다고 무조건 좋은 것이 아니다. 운영 환경에서는 어느 축을 더 중요하게 볼지 제품 목적이 먼저 정해져야 한다.

### 데이터 복잡도가 모델 차이를 증폭시키는 방식

grouped bar chart는 같은 모델이라도 데이터가 바뀌면 성능이 얼마나 달라지는지 보여준다. 보통 `demo`에서는 9B와 2B의 차이가 작고, `tech_docs`에서는 그 차이가 더 커진다. 이유는 모델이 갑자기 똑똑해졌다기보다, 큰 코퍼스에서 더 많은 chunk를 읽고 조합해야 하므로 reasoning·압축·표현 능력 차이가 드러나기 때문이다.

그래서 모델 선택은 파라미터 수만 보고 결정하면 안 된다. 반드시 "내 데이터가 얼마나 복잡한가"를 함께 봐야 한다.

**💡 면접 포인트**
- 단순 문서에서는 2B도 충분해 비용을 아낄 수 있지만, 복잡한 기술 문서에서는 9B가 품질 우위를 보일 수 있다.
- 같은 모델도 데이터 복잡도에 따라 성능 차이가 크게 달라지므로, 모델 선택은 데이터 특성과 함께 결정해야 한다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chart_summary = model_summary.copy()
latency_min = chart_summary['latency_seconds'].min()
latency_max = chart_summary['latency_seconds'].max()
if latency_max == latency_min:
    chart_summary['speed_score'] = 1.0
else:
    chart_summary['speed_score'] = 1 - (
        (chart_summary['latency_seconds'] - latency_min) / (latency_max - latency_min)
    )

variant_order = ['qwen3.5:9b', 'qwen3.5:4b', 'qwen3.5:2b', 'rule_based']
dataset_order = ['demo', 'tech_docs']
color_map = {'demo': '#4C78A8', 'tech_docs': '#F58518'}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, metric, title in [
    (axes[0], 'answer_correctness', 'Answer Correctness by Dataset'),
    (axes[1], 'latency_seconds', 'Latency by Dataset'),
]:
    pivot = chart_summary.pivot(index='variant', columns='dataset', values=metric)
    pivot = pivot.reindex(variant_order)
    present_datasets = [name for name in dataset_order if name in pivot.columns]
    pivot = pivot[present_datasets]
    pivot.plot(kind='bar', ax=ax, color=[color_map[name] for name in present_datasets], width=0.75)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
    if metric == 'answer_correctness':
        ax.set_ylim(0, 1)
    ax.legend(title='dataset')
plt.tight_layout()
plt.show()

radar_metrics = ['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'speed_score']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

present_datasets = [name for name in dataset_order if name in set(chart_summary['dataset'])]
fig, axes = plt.subplots(1, len(present_datasets), figsize=(7 * len(present_datasets), 6), subplot_kw={'polar': True})
if len(present_datasets) == 1:
    axes = [axes]

for ax, dataset_name in zip(axes, present_datasets):
    subset = chart_summary[chart_summary['dataset'] == dataset_name].set_index('variant').reindex(variant_order).reset_index()
    for _, row in subset.dropna(subset=['variant']).iterrows():
        values = [row[metric] for metric in radar_metrics]
        values += values[:1]
        ax.plot(angles, values, label=row['variant'])
        ax.fill(angles, values, alpha=0.06)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_metrics)
    ax.set_ylim(0, 1)
    ax.set_title(f'{dataset_name} Radar')
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()

comparison_gap


## 메모리 사용량 비교

**목적**
- 모델 품질과 속도뿐 아니라, DGX에서 실제 GPU 메모리를 얼마나 점유하는지도 함께 본다.

**핵심 로직**
- `nvidia-smi --query-compute-apps=process_name,used_gpu_memory`를 호출해 Ollama 프로세스의 GPU 메모리 사용량을 스냅샷으로 읽는다.
- rule-based는 LLM을 전혀 호출하지 않으므로 메모리 0으로 기록한다.

**주요 파라미터**
- `gpu_memory_mb_snapshot`: 해당 시점 Ollama 프로세스 메모리 사용량이다.
- `measurement`: 실제 측정 성공인지, 서버 부재인지, error인지 알려 주는 해석용 라벨이다.

**결과 해석 가이드**
- 큰 모델일수록 보통 GPU 메모리 사용량이 크고, latency도 길어질 가능성이 높다.
- `No Ollama GPU process detected`가 나오면 모델이 CPU에서 돌았거나, 호출 직후 프로세스가 내려간 것일 수 있다.
- memory snapshot은 순간값이므로 절대 지표보다는 모델 간 상대 비교로 읽는 편이 안전하다.

**💡 면접 포인트**
- 모델 선택은 품질만의 문제가 아니라 VRAM budget과 latency budget의 문제다.
- DGX처럼 GPU가 넉넉한 환경에서도 운영 비용과 동시성 요구를 고려하면 무조건 9B가 정답은 아니다.


In [ ]:
import shutil
import subprocess
import time

import pandas as pd

from src.llm_client import LLMConfig, OllamaClient

def ollama_gpu_memory_snapshot() -> tuple[str, float | None]:
    if shutil.which('nvidia-smi') is None:
        return 'nvidia-smi unavailable', None
    command = [
        'nvidia-smi',
        '--query-compute-apps=process_name,used_gpu_memory',
        '--format=csv,noheader,nounits',
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        return result.stderr.strip() or 'nvidia-smi query failed', None
    total = 0.0
    matched = False
    for line in result.stdout.splitlines():
        parts = [part.strip() for part in line.split(',')]
        if len(parts) != 2:
            continue
        process_name, used_memory = parts
        if 'ollama' in process_name.lower():
            matched = True
            try:
                total += float(used_memory)
            except ValueError:
                continue
    if matched:
        return 'ok', round(total, 1)
    return 'No Ollama GPU process detected', None

memory_rows = []
for variant in ['qwen3.5:9b', 'qwen3.5:4b', 'qwen3.5:2b', 'rule_based']:
    if variant == 'rule_based':
        memory_rows.append(
            {
                'variant': variant,
                'measurement': 'rule_based_no_llm',
                'gpu_memory_mb_snapshot': 0.0,
            }
        )
        continue

    if not llm_live:
        memory_rows.append(
            {
                'variant': variant,
                'measurement': 'server_unavailable',
                'gpu_memory_mb_snapshot': None,
            }
        )
        continue

    try:
        client = OllamaClient(LLMConfig(model=variant, max_tokens=32, timeout=30))
        client.chat(
            [
                {'role': 'system', 'content': 'Answer briefly.'},
                {'role': 'user', 'content': 'Return one short sentence about grounded answers.'},
            ]
        )
        time.sleep(1.0)
        measurement, memory_mb = ollama_gpu_memory_snapshot()
    except Exception as error:
        measurement, memory_mb = f'fallback_or_error: {error}', None

    memory_rows.append(
        {
            'variant': variant,
            'measurement': measurement,
            'gpu_memory_mb_snapshot': memory_mb,
        }
    )

pd.DataFrame(memory_rows)

## 핵심 정리

이 노트북을 통해 모델 크기 선택은 "큰 모델이 더 좋다"가 아니라, 답변 품질(answer correctness), 근거 신뢰도(grounding), 지연 시간(latency), GPU 메모리 사용량(VRAM) 사이의 트레이드오프라는 점을 확인했다. 9B는 보통 품질과 표현력에서 유리하지만 비용이 크고, 2B는 빠르고 가볍지만 복잡한 질문에서 coverage가 약해질 수 있다. 규칙 기반 경로는 가장 저렴하지만 답변 표현력과 복합 reasoning 한계가 분명하다.

**💡 면접 포인트**
- 모델 크기별 trade-off를 실험으로 확인했고, 프로토타이핑/검증/운영에 따라 선택 기준을 다르게 제시할 수 있다.
- 품질 비교는 평균표만이 아니라 단일 질문 질적 비교, live/fallback 비율, GPU 메모리까지 함께 봐야 현실적이다.
- 같은 workflow contract 위에서 모델만 바꾸면 시스템 효과와 모델 효과를 분리해 설명하기 쉽다.

데이터 복잡도를 함께 보면 모델 선택 기준이 더 현실적으로 바뀐다. `demo`처럼 작은 코퍼스에서는 규칙 기반과 2B 모델도 강한 baseline이 될 수 있지만, `tech_docs`에서는 4B와 9B가 더 분명한 품질 우위를 보일 수 있다. 즉 "어떤 모델이 최고인가"보다 "어떤 데이터에서 어느 모델까지 투자할 가치가 있는가"가 더 중요한 질문이다.

**💡 면접 포인트**
- 모델 비교는 파라미터 수만이 아니라 데이터 복잡도와 함께 봐야 한다.
- 작은 데이터에서는 경량 모델로 비용을 아끼고, 복잡한 실제 데이터에서는 큰 모델로 품질을 확보하는 식의 운영 전략을 제안할 수 있다.
